# Flow Matching (Euler) on CIFAR-10

Dedicated CIFAR-10 notebook (separate from STL10), with training metrics and generation metrics saved during training.

In [1]:
# Optional in Colab
!pip install -q torch torchvision torchmetrics torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.2 MB/s eta 0:00:00


In [2]:
import copy
import json
import math
import random
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore
    TORCHMETRICS_OK = True
except Exception as e:
    TORCHMETRICS_OK = False
    TORCHMETRICS_ERR = str(e)

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CFM_Euler10_conditional')
else:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / 'data'
RUN_ROOT = PROJECT_ROOT / 'runs'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('torchmetrics:', 'OK' if TORCHMETRICS_OK else f'MISSING ({TORCHMETRICS_ERR})')

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/CFM_Euler10_conditional
DATA_ROOT: /content/drive/MyDrive/CFM_Euler10_conditional/data
RUN_ROOT: /content/drive/MyDrive/CFM_Euler10_conditional/runs
torchmetrics: OK


In [3]:
@dataclass
class Config:
    dataset_name: str = 'cifar10'
    image_size: int = 32
    batch_size: int = 128
    num_workers: int = 2
    lr: float = 2e-4
    weight_decay: float = 1e-5
    epochs: int = 500
    print_every: int = 100
    save_every_epoch: int = 1
    run_prefix: str = 'cfm_cifar10_conditional'

    use_ema: bool = True
    ema_decay: float = 0.999

    # model capacity
    base_ch: int = 128
    t_dim: int = 256
    num_classes: int = 10
    class_dropout_p: float = 0.1
    use_attn_16: bool = True
    use_attn_8: bool = True

    # progress images during training
    progress_nfe: int = 50
    progress_num_images: int = 64
    fixed_vis_seed: int = 123
    progress_cfg_scale: float = 1.5

    # generation eval during training
    eval_every_epoch: int = 10
    eval_nfes: tuple = (10, 20, 50, 100)
    eval_num_images: int = 10000
    eval_batch_size: int = 100
    eval_seeds: tuple = (2026, 2027, 2028)
    eval_cfg_scale: float = 1.0
    save_eval_samples: bool = False

cfg = Config()
print(cfg)


Config(dataset_name='cifar10', image_size=32, batch_size=128, num_workers=2, lr=0.0002, weight_decay=1e-05, epochs=500, print_every=100, save_every_epoch=1, run_prefix='cfm_cifar10_conditional', use_ema=True, ema_decay=0.999, base_ch=128, t_dim=256, num_classes=10, class_dropout_p=0.1, use_attn_16=True, use_attn_8=True, progress_nfe=50, progress_num_images=64, fixed_vis_seed=123, progress_cfg_scale=1.5, eval_every_epoch=10, eval_nfes=(10, 20, 50, 100), eval_num_images=10000, eval_batch_size=100, eval_seeds=(2026, 2027, 2028), eval_cfg_scale=1.0, save_eval_samples=False)


In [4]:
# Download CIFAR-10 (train + test)
datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=True)
datasets.CIFAR10(root=str(DATA_ROOT), train=False, download=True)
print('CIFAR-10 downloaded to:', DATA_ROOT)

100%|██████████| 170M/170M [00:02<00:00, 79.8MB/s] 


CIFAR-10 downloaded to: /content/drive/MyDrive/CFM_Euler10_conditional/data


In [4]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [5]:
transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = datasets.CIFAR10(
    root=str(DATA_ROOT),
    train=True,
    transform=transform,
    download=True,
)

train_loader = DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

eval_real_loader = DataLoader(
    train_set,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print('train samples:', len(train_set), '| steps/epoch:', len(train_loader))

train samples: 50000 | steps/epoch: 390


In [6]:
def sinusoidal_time_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half - 1, 1))
    angles = t[:, None] * freqs[None, :] * 2 * math.pi
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


class SelfAttention2d(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.q = nn.Conv2d(ch, ch, kernel_size=1)
        self.k = nn.Conv2d(ch, ch, kernel_size=1)
        self.v = nn.Conv2d(ch, ch, kernel_size=1)
        self.proj = nn.Conv2d(ch, ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        h_ = self.norm(x)

        q = self.q(h_).reshape(b, c, h * w).permute(0, 2, 1)  # [B, HW, C]
        k = self.k(h_).reshape(b, c, h * w)                    # [B, C, HW]
        attn = torch.bmm(q, k) * (c ** -0.5)
        attn = torch.softmax(attn, dim=-1)

        v = self.v(h_).reshape(b, c, h * w).permute(0, 2, 1)  # [B, HW, C]
        out = torch.bmm(attn, v).permute(0, 2, 1).reshape(b, c, h, w)
        return x + self.proj(out)


class TimeResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.y_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor, y_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x)
        h = self.norm1(h)
        h = h + self.t_proj(t_emb)[:, :, None, None] + self.y_proj(y_emb)[:, :, None, None]
        h = F.silu(h)
        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h + self.skip(x)


class SimpleFlowUNet(nn.Module):
    def __init__(
        self,
        in_ch: int = 3,
        base_ch: int = 128,
        t_dim: int = 256,
        num_classes: int = 10,
        use_attn_16: bool = True,
        use_attn_8: bool = True,
    ):
        super().__init__()
        self.t_dim = t_dim
        self.num_classes = num_classes
        self.null_label = num_classes

        self.time_mlp = nn.Sequential(
            nn.Linear(t_dim, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.class_emb = nn.Embedding(num_classes + 1, t_dim)

        # 32x32
        self.enc1a = TimeResBlock(in_ch, base_ch, t_dim)
        self.enc1b = TimeResBlock(base_ch, base_ch, t_dim)

        # 16x16
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)
        self.enc2a = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.enc2_attn1 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()
        self.enc2b = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.enc2_attn2 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()

        # 8x8
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1)
        self.enc3a = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.enc3_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()
        self.enc3b = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)

        self.mid1 = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.mid_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()
        self.mid2 = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)

        # decoder
        self.dec3a = TimeResBlock(base_ch * 8, base_ch * 4, t_dim)
        self.dec3b = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.dec3_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()

        self.dec2a = TimeResBlock(base_ch * 6, base_ch * 2, t_dim)
        self.dec2_attn1 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()
        self.dec2b = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.dec2_attn2 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()

        self.dec1a = TimeResBlock(base_ch * 3, base_ch, t_dim)
        self.dec1b = TimeResBlock(base_ch, base_ch, t_dim)
        self.out = nn.Conv2d(base_ch, in_ch, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        if y is None:
            y = torch.full((x.size(0),), self.null_label, device=x.device, dtype=torch.long)

        t_emb = self.time_mlp(sinusoidal_time_embedding(t, self.t_dim))
        y_emb = self.class_emb(y.long())

        s1 = self.enc1a(x, t_emb, y_emb)
        s1 = self.enc1b(s1, t_emb, y_emb)

        h = self.down1(s1)
        s2 = self.enc2a(h, t_emb, y_emb)
        s2 = self.enc2_attn1(s2)
        s2 = self.enc2b(s2, t_emb, y_emb)
        s2 = self.enc2_attn2(s2)

        h = self.down2(s2)
        s3 = self.enc3a(h, t_emb, y_emb)
        s3 = self.enc3_attn(s3)
        s3 = self.enc3b(s3, t_emb, y_emb)

        h = self.mid1(s3, t_emb, y_emb)
        h = self.mid_attn(h)
        h = self.mid2(h, t_emb, y_emb)

        h = torch.cat([h, s3], dim=1)
        h = self.dec3a(h, t_emb, y_emb)
        h = self.dec3b(h, t_emb, y_emb)
        h = self.dec3_attn(h)

        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = torch.cat([h, s2], dim=1)
        h = self.dec2a(h, t_emb, y_emb)
        h = self.dec2_attn1(h)
        h = self.dec2b(h, t_emb, y_emb)
        h = self.dec2_attn2(h)

        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = torch.cat([h, s1], dim=1)
        h = self.dec1a(h, t_emb, y_emb)
        h = self.dec1b(h, t_emb, y_emb)
        return self.out(h)


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model: nn.Module):
        for p_s, p in zip(self.shadow.parameters(), model.parameters()):
            p_s.data.mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)


@torch.no_grad()
def euler_sample(
    model: nn.Module,
    n_samples: int,
    image_size: int,
    steps: int,
    seed: int,
    device: torch.device,
    labels: torch.Tensor | None = None,
    cfg_scale: float = 1.0,
):
    model.eval()
    gen = torch.Generator(device=device.type)
    gen.manual_seed(seed)

    x = torch.randn(n_samples, 3, image_size, image_size, device=device, generator=gen)

    if labels is None:
        labels = torch.randint(0, model.num_classes, (n_samples,), device=device, generator=gen)
    else:
        labels = labels.to(device).long()
        if labels.ndim == 0:
            labels = labels.repeat(n_samples)
        if labels.shape[0] != n_samples:
            raise ValueError(f'labels length ({labels.shape[0]}) must equal n_samples ({n_samples})')

    null_labels = torch.full_like(labels, model.null_label)

    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((n_samples,), k / steps, device=device)
        v_cond = model(x, t, labels)
        if cfg_scale != 1.0:
            v_uncond = model(x, t, null_labels)
            v = v_uncond + cfg_scale * (v_cond - v_uncond)
        else:
            v = v_cond
        x = x + dt * v
    return x.clamp(-1, 1)


def to_uint8(x_minus1_1: torch.Tensor) -> torch.Tensor:
    x01 = ((x_minus1_1 + 1.0) / 2.0).clamp(0, 1)
    return (x01 * 255).to(torch.uint8)


def append_jsonl(path: Path, record: dict):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record) + '\n')


def save_eval_snapshot(metrics_dir: Path, eval_rows: list, fid_by_epoch: dict, runtime_by_epoch: dict):
    with open(metrics_dir / 'eval_metrics_summary.json', 'w', encoding='utf-8') as f:
        json.dump(eval_rows, f, indent=2)
    with open(metrics_dir / 'eval_fid_results.json', 'w', encoding='utf-8') as f:
        json.dump(fid_by_epoch, f, indent=2)
    with open(metrics_dir / 'eval_runtime_results.json', 'w', encoding='utf-8') as f:
        json.dump(runtime_by_epoch, f, indent=2)


@torch.no_grad()
def run_eval_epoch(
    epoch: int,
    sample_model: nn.Module,
    metrics_dir: Path,
    eval_rows: list,
    fid_by_epoch: dict,
    runtime_by_epoch: dict,
    eval_jsonl_path: Path,
):
    if not TORCHMETRICS_OK:
        print('Skipping generation metrics (torchmetrics missing).')
        return

    epoch_key = f'epoch_{epoch:03d}'
    fid_by_epoch.setdefault(epoch_key, {})
    runtime_by_epoch.setdefault(epoch_key, {})

    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False, reset_real_features=False).to(device)
    real_seen = 0
    for real_x, _ in eval_real_loader:
        real_x = real_x.to(device)
        fid_metric.update(to_uint8(real_x), real=True)
        real_seen += real_x.size(0)
        if real_seen >= cfg.eval_num_images:
            break

    print(f'[Eval epoch {epoch:03d}] real images cached: {min(real_seen, cfg.eval_num_images)}')

    for nfe in cfg.eval_nfes:
        fid_values = []
        is_mean_values = []
        is_std_values = []
        sec_per_img_values = []
        total_sec_values = []

        for eval_seed in cfg.eval_seeds:
            fid_metric.reset()
            is_metric = InceptionScore(normalize=False).to(device)

            generated = 0
            batch_idx = 0
            t0 = time.time()

            sample_dir = metrics_dir.parent / 'samples_eval' / f'epoch_{epoch:03d}' / f'nfe_{nfe}' / f'seed_{eval_seed}'
            if cfg.save_eval_samples:
                sample_dir.mkdir(parents=True, exist_ok=True)

            while generated < cfg.eval_num_images:
                cur_bs = min(cfg.eval_batch_size, cfg.eval_num_images - generated)
                x_fake = euler_sample(
                    sample_model,
                    n_samples=cur_bs,
                    image_size=cfg.image_size,
                    steps=int(nfe),
                    seed=eval_seed + epoch * 100000 + int(nfe) * 1000 + batch_idx,
                    device=device,
                    labels=None,
                    cfg_scale=cfg.eval_cfg_scale,
                )

                fake_u8 = to_uint8(x_fake)
                fid_metric.update(fake_u8, real=False)
                is_metric.update(fake_u8)

                if cfg.save_eval_samples:
                    x01 = ((x_fake + 1.0) / 2.0).clamp(0, 1).cpu()
                    for i in range(cur_bs):
                        img_idx = generated + i
                        save_image(x01[i], sample_dir / f'img_{img_idx:06d}.png')

                generated += cur_bs
                batch_idx += 1

            total_sec = time.time() - t0
            sec_per_img = total_sec / max(generated, 1)
            fid_value = float(fid_metric.compute().item())
            is_mean_t, is_std_t = is_metric.compute()
            is_mean = float(is_mean_t.item())
            is_std = float(is_std_t.item())

            fid_values.append(fid_value)
            is_mean_values.append(is_mean)
            is_std_values.append(is_std)
            sec_per_img_values.append(sec_per_img)
            total_sec_values.append(total_sec)

            print(
                f'[Eval epoch {epoch:03d}] NFE {nfe} seed {eval_seed}: '
                f'FID={fid_value:.6f}, sec/img={sec_per_img:.6f}'
            )

        fid_mean = float(np.mean(fid_values))
        fid_std = float(np.std(fid_values))
        is_mean_mean = float(np.mean(is_mean_values))
        is_mean_std = float(np.std(is_mean_values))
        is_std_mean = float(np.mean(is_std_values))
        sec_per_img_mean = float(np.mean(sec_per_img_values))
        total_sec_mean = float(np.mean(total_sec_values))

        row = {
            'epoch': int(epoch),
            'model': 'CFM_EMA_CONDITIONAL' if cfg.use_ema else 'CFM_CONDITIONAL',
            'nfe': int(nfe),
            'num_generated_per_seed': int(cfg.eval_num_images),
            'num_seeds': int(len(cfg.eval_seeds)),
            'fid': fid_mean,
            'fid_std': fid_std,
            'is_mean': is_mean_mean,
            'is_mean_std': is_mean_std,
            'is_std': is_std_mean,
            'sec_per_image': sec_per_img_mean,
            'total_sampling_sec': total_sec_mean,
            'seeds': [int(s) for s in cfg.eval_seeds],
            'sampler': 'euler',
            'solver': 'explicit_euler',
            'cfg_scale': float(cfg.eval_cfg_scale),
        }
        eval_rows.append(row)
        append_jsonl(eval_jsonl_path, row)

        fid_by_epoch[epoch_key][f'nfe_{nfe}'] = {
            'fid_mean': fid_mean,
            'fid_std': fid_std,
            'is_mean_mean': is_mean_mean,
            'is_mean_std': is_mean_std,
            'is_std_mean': is_std_mean,
            'seeds': [int(s) for s in cfg.eval_seeds],
        }
        runtime_by_epoch[epoch_key][f'nfe_{nfe}'] = {
            'num_generated_per_seed': int(cfg.eval_num_images),
            'num_seeds': int(len(cfg.eval_seeds)),
            'total_sampling_sec_mean': total_sec_mean,
            'sec_per_image_mean': sec_per_img_mean,
        }

        save_eval_snapshot(metrics_dir, eval_rows, fid_by_epoch, runtime_by_epoch)
        print(
            f'[Eval epoch {epoch:03d}] NFE {nfe}: '
            f'FID(mean?std)={fid_mean:.6f}?{fid_std:.6f}, '
            f'sec/img(mean)={sec_per_img_mean:.6f}'
        )


In [7]:
model = SimpleFlowUNet(
    in_ch=3,
    base_ch=cfg.base_ch,
    t_dim=cfg.t_dim,
    num_classes=cfg.num_classes,
    use_attn_16=cfg.use_attn_16,
    use_attn_8=cfg.use_attn_8,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = RUN_ROOT / f"{cfg.run_prefix}_{run_id}"
checkpoints_dir = run_dir / 'checkpoints'
progress_dir = run_dir / 'samples'
metrics_dir = run_dir / 'metrics'
for d in [run_dir, checkpoints_dir, progress_dir, metrics_dir]:
    d.mkdir(parents=True, exist_ok=True)

with open(run_dir / 'config.json', 'w', encoding='utf-8') as f:
    payload = {
        'config': asdict(cfg),
        'run_id': run_id,
        'created_at': datetime.now().isoformat(),
        'seed': seed,
    }
    json.dump(payload, f, indent=2)

history = []
best_loss = float('inf')

train_metrics_jsonl_path = metrics_dir / 'training_metrics.jsonl'
eval_metrics_jsonl_path = metrics_dir / 'eval_metrics_summary.jsonl'
train_metrics_jsonl_path.write_text('', encoding='utf-8')
eval_metrics_jsonl_path.write_text('', encoding='utf-8')

eval_rows = []
fid_results_by_epoch = {}
runtime_results_by_epoch = {}

progress_labels = torch.arange(cfg.num_classes, device=device)
progress_labels = progress_labels.repeat((cfg.progress_num_images + cfg.num_classes - 1) // cfg.num_classes)
progress_labels = progress_labels[:cfg.progress_num_images]

print('Run directory:', run_dir)


Run directory: /content/drive/MyDrive/CFM_Euler10_conditional/runs/cfm_cifar10_conditional_20260311_182506


In [ ]:
# Resume setup (run this instead of creating a new run)
from pathlib import Path
import json
import torch

# 1) Point to existing run
run_dir = Path("/content/drive/MyDrive/CFM_Euler10_conditional/runs/cfm_cifar10_conditional_20260311_162300")
checkpoints_dir = run_dir / "checkpoints"
metrics_dir = run_dir / "metrics"
progress_dir = run_dir / "samples"

ckpt = torch.load(checkpoints_dir / "last.pt", map_location=device)

# 2) Rebuild model/optimizer/ema exactly as before
model = SimpleFlowUNet(
    in_ch=3,
    base_ch=cfg.base_ch,
    t_dim=cfg.t_dim,
    num_classes=cfg.num_classes,
    use_attn_16=cfg.use_attn_16,
    use_attn_8=cfg.use_attn_8,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None

model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
if ema is not None and ckpt.get("ema_state_dict") is not None:
    ema.shadow.load_state_dict(ckpt["ema_state_dict"])

# 3) Resume epoch/history
start_epoch = int(ckpt["epoch"]) + 1
run_id = ckpt.get("run_id", run_dir.name)

tm_path = metrics_dir / "training_metrics_latest.json"
if tm_path.exists():
    tm = json.loads(tm_path.read_text(encoding="utf-8"))
    history = tm.get("history", [])
    best_loss = float(min(history)) if len(history) else float("inf")
else:
    history, best_loss = [], float("inf")

# Keep appending to same logs
train_metrics_jsonl_path = metrics_dir / "training_metrics.jsonl"
eval_metrics_jsonl_path = metrics_dir / "eval_metrics_summary.jsonl"

print("Resuming run:", run_dir)
print("Start epoch:", start_epoch)


In [8]:
from pathlib import Path
import json

# run_dir déjà défini dans ta cellule resume
metrics_dir = run_dir / "metrics"
checkpoints_dir = run_dir / "checkpoints"
progress_dir = run_dir / "samples"

train_metrics_jsonl_path = metrics_dir / "training_metrics.jsonl"
eval_metrics_jsonl_path = metrics_dir / "eval_metrics_summary.jsonl"

# Charger les historiques existants (ou init vides)
eval_json = metrics_dir / "eval_metrics_summary.json"
fid_json = metrics_dir / "eval_fid_results.json"
runtime_json = metrics_dir / "eval_runtime_results.json"

eval_rows = json.loads(eval_json.read_text(encoding="utf-8")) if eval_json.exists() else []
fid_results_by_epoch = json.loads(fid_json.read_text(encoding="utf-8")) if fid_json.exists() else {}
runtime_results_by_epoch = json.loads(runtime_json.read_text(encoding="utf-8")) if runtime_json.exists() else {}

# labels de visu
progress_labels = torch.arange(cfg.num_classes, device=device)
progress_labels = progress_labels.repeat(
    (cfg.progress_num_images + cfg.num_classes - 1) // cfg.num_classes
)[:cfg.progress_num_images]

print("Resume context OK")
print("eval_rows:", len(eval_rows))


Resume context OK
eval_rows: 40


In [9]:
for epoch in range(start_epoch, cfg.epochs + 1):
    model.train()
    running = 0.0
    t_epoch = time.time()

    for step, (x0, y) in enumerate(train_loader, start=1):
        x0 = x0.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).long()
        b = x0.size(0)

        z = torch.randn_like(x0)
        t = torch.rand(b, device=device)
        xt = (1.0 - t)[:, None, None, None] * z + t[:, None, None, None] * x0
        target = x0 - z

        if cfg.class_dropout_p > 0.0:
            y_in = y.clone()
            drop_mask = torch.rand(b, device=device) < cfg.class_dropout_p
            y_in[drop_mask] = model.null_label
        else:
            y_in = y

        pred = model(xt, t, y_in)
        loss = F.mse_loss(pred, target)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if ema is not None:
            ema.update(model)

        running += loss.item()

        if step % cfg.print_every == 0:
            print(f"Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | Loss {running/step:.4f}")

    epoch_loss = running / len(train_loader)
    history.append(epoch_loss)
    epoch_time = time.time() - t_epoch

    epoch_record = {
        'epoch': int(epoch),
        'epoch_loss': float(epoch_loss),
        'epoch_time_sec': float(epoch_time),
        'best_epoch_loss': float(min(history)),
    }
    append_jsonl(train_metrics_jsonl_path, epoch_record)

    with open(metrics_dir / 'training_metrics_latest.json', 'w', encoding='utf-8') as f:
        json.dump({'history': history, **epoch_record}, f, indent=2)

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'ema_state_dict': ema.shadow.state_dict() if ema is not None else None,
        'config': asdict(cfg),
        'run_id': run_id,
        'epoch_loss': float(epoch_loss),
    }
    torch.save(ckpt, checkpoints_dir / 'last.pt')
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(ckpt, checkpoints_dir / 'best.pt')
    if epoch % cfg.save_every_epoch == 0:
        torch.save(ckpt, checkpoints_dir / f'epoch_{epoch:03d}.pt')

    sample_model = ema.shadow if ema is not None else model
    x_vis = euler_sample(
        sample_model,
        cfg.progress_num_images,
        cfg.image_size,
        cfg.progress_nfe,
        cfg.fixed_vis_seed,
        device,
        labels=progress_labels,
        cfg_scale=cfg.progress_cfg_scale,
    )
    x_vis = ((x_vis + 1.0) / 2.0).clamp(0, 1).cpu()
    save_image(x_vis, progress_dir / f'progress_epoch_{epoch:03d}.png', nrow=8)

    do_eval = cfg.eval_every_epoch > 0 and (epoch % cfg.eval_every_epoch == 0 or epoch == cfg.epochs)
    if do_eval:
        run_eval_epoch(
            epoch=epoch,
            sample_model=sample_model,
            metrics_dir=metrics_dir,
            eval_rows=eval_rows,
            fid_by_epoch=fid_results_by_epoch,
            runtime_by_epoch=runtime_results_by_epoch,
            eval_jsonl_path=eval_metrics_jsonl_path,
        )

    print(f"[Epoch {epoch:03d}] mean loss={epoch_loss:.4f} | time={epoch_time:.1f}s")

print('Training finished. Run directory:', run_dir)
print('Training metrics:', metrics_dir / 'training_metrics_latest.json')
print('Generation metrics:', metrics_dir / 'eval_metrics_summary.json')


Epoch 131 | Step 0100/390 | Loss 0.1694
Epoch 131 | Step 0200/390 | Loss 0.1681
Epoch 131 | Step 0300/390 | Loss 0.1680
[Epoch 131] mean loss=0.1681 | time=46.6s
Epoch 132 | Step 0100/390 | Loss 0.1684
Epoch 132 | Step 0200/390 | Loss 0.1690
Epoch 132 | Step 0300/390 | Loss 0.1689
[Epoch 132] mean loss=0.1687 | time=44.8s
Epoch 133 | Step 0100/390 | Loss 0.1696
Epoch 133 | Step 0200/390 | Loss 0.1685
Epoch 133 | Step 0300/390 | Loss 0.1684
[Epoch 133] mean loss=0.1681 | time=44.6s
Epoch 134 | Step 0100/390 | Loss 0.1695
Epoch 134 | Step 0200/390 | Loss 0.1690
Epoch 134 | Step 0300/390 | Loss 0.1687
[Epoch 134] mean loss=0.1685 | time=44.7s
Epoch 135 | Step 0100/390 | Loss 0.1716
Epoch 135 | Step 0200/390 | Loss 0.1706
Epoch 135 | Step 0300/390 | Loss 0.1707
[Epoch 135] mean loss=0.1708 | time=44.6s
Epoch 136 | Step 0100/390 | Loss 0.1693
Epoch 136 | Step 0200/390 | Loss 0.1701
Epoch 136 | Step 0300/390 | Loss 0.1702
[Epoch 136] mean loss=0.1702 | time=44.6s
Epoch 137 | Step 0100/390 | 

Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 247MB/s]


[Eval epoch 140] real images cached: 10000


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Eval epoch 140] NFE 10 seed 2026: FID=21.972181, sec/img=0.003376
[Eval epoch 140] NFE 10 seed 2027: FID=21.976791, sec/img=0.003370
[Eval epoch 140] NFE 10 seed 2028: FID=21.990017, sec/img=0.003371
[Eval epoch 140] NFE 10: FID(mean?std)=21.979663?0.007559, sec/img(mean)=0.003372
[Eval epoch 140] NFE 20 seed 2026: FID=17.579819, sec/img=0.005849
[Eval epoch 140] NFE 20 seed 2027: FID=17.575354, sec/img=0.005851
[Eval epoch 140] NFE 20 seed 2028: FID=17.586082, sec/img=0.005850
[Eval epoch 140] NFE 20: FID(mean?std)=17.580418?0.004400, sec/img(mean)=0.005850
[Eval epoch 140] NFE 50 seed 2026: FID=14.809010, sec/img=0.013286
[Eval epoch 140] NFE 50 seed 2027: FID=14.809867, sec/img=0.013306
[Eval epoch 140] NFE 50 seed 2028: FID=14.786886, sec/img=0.013291
[Eval epoch 140] NFE 50: FID(mean?std)=14.801921?0.010637, sec/img(mean)=0.013294
[Eval epoch 140] NFE 100 seed 2026: FID=14.074348, sec/img=0.025694
[Eval epoch 140] NFE 100 seed 2027: FID=14.100781, sec/img=0.025740
[Eval epoch 140

KeyboardInterrupt: 

In [ ]:
# Quick visual check
sample_model = ema.shadow if (cfg.use_ema and ema is not None) else model
vis_labels = torch.arange(cfg.num_classes, device=device)
vis_labels = vis_labels.repeat((64 + cfg.num_classes - 1) // cfg.num_classes)[:64]

x = euler_sample(
    sample_model,
    n_samples=64,
    image_size=cfg.image_size,
    steps=100,
    seed=2026,
    device=device,
    labels=vis_labels,
    cfg_scale=cfg.progress_cfg_scale,
)
x = ((x + 1.0) / 2.0).clamp(0, 1).cpu()
grid = make_grid(x, nrow=8)
save_image(grid, run_dir / 'final_grid.png')
print('Saved:', run_dir / 'final_grid.png')
print('Class order in grid (row-major):', vis_labels.cpu().tolist())


In [ ]:
# Optional quick summary after training
import pandas as pd

eval_json = metrics_dir / 'eval_metrics_summary.json'
if eval_json.exists():
    df = pd.DataFrame(json.loads(eval_json.read_text(encoding='utf-8')))
    if len(df) > 0:
        cols = ['epoch', 'nfe', 'fid', 'fid_std', 'is_mean', 'is_mean_std', 'sec_per_image', 'num_seeds']
        cols = [c for c in cols if c in df.columns]
        display(df.sort_values(['epoch', 'nfe'])[cols])
        display(df.pivot(index='nfe', columns='epoch', values='fid'))
else:
    print('No eval file found yet.')
